# Research Proposal
___
___

**A Deep Reinforcement Learning Approach to a Systematic Options Overlay Strategy using Arbitrage-Free Option Surfaces**

*This proposal builds a pipeline that fits real option surfaces with SANOS, a non-parametric method guaranteeing smooth, strictly arbitrage-free prices via convex Black-Scholes kernels, then extends it into a dynamic generative model (DYSANOS) by evolving a low-dimensional latent surface state, simulated jointly with the underlying spot price. The resulting realistic, always arbitrage-free simulated paths train a deep hedging reinforcement-learning agent to learn optimal options rebalancing (timing, sizing, strike selection) that maximizes a trade-off between expected wealth and risk (e.g. CVaR). Open challenges include multi-asset extensions in deep hedging, adjusting the deep hedging algorithm for an investment strategy, and efficient time-series modeling of the spot and option surfaces jointly.*

**KEYWORDS**: Option Surface Modeling, Deep Hedging, Reinforcement Learning, Time Series Analysis, Dynamic Portfolio Management 


___
___
### **1. Benchmark**


### Dynamic Portfolio Management with Financial Derivatives

D. Barro, G. Consigli, V. Varun. A stochastic programming model for dynamic portfolio management with financial derivatives. Journal of Banking and Finance, Volume 140, 2022. https://doi.org/10.1016/j.jbankfin.2022.106445 

*The paper builds an arbitrage-free scenario tree by bootstrapping historical joint returns of the relevant assets, preserving their empirical correlations. Derivatives are then priced on that tree via a minimal-entropy risk-neutral measure and backward induction. With all prices fixed, a linear program solves for the trading policy (asset and derivative positions at each node) that maximizes a tunable trade-off between expected wealth and tail risk, subject to wealth and position constraints. This is re-solved on a rolling basis as new data arrives, so the policy adapts to the current regime.*

*Empirically, results are strong; its strengths are an exact, interpretable optimum with hard constraints enforced, and regime-adaptive sizing; its main weaknesses are that the policy is only optimal for the specific tree solved, is sensitive to how well bootstrapping represents future markets, scales poorly with dimension, and must be re-solved from scratch each period rather than yielding a reusable policy.*


**Summary**

1. Build the tree
    - Each node knows 4 stochastic values (equity, bond, money market, VIX) + a risk free rate (deterministic) used separately.
    - The 4 stochastic assets get bootstrapped to build the tree: for each node, a children is created by sampling a historical return vectors of these assets: to keep there historical correlation. 
    - Each node must have at least 4 children to solve non-arbitrage conditions. The paper uses monthly nodes for 6 months, and the first node has 10 childen, that is, total number of nodes is $\small 1 + 10 + 10 \ (4) + 10 \ (4^2) + 10 \ (4^3) + ...$
    - Then, we have a full arbitrage-free tree of asset prices under the physical measure.


2. Price derivatives on that tree
    - Once we have the full arbitrage-free tree of asset prices with physical measure $p(n)$, fit the minimal-entropy risk-neutral measure $q(n)$ node-by-node (same tree, different probabilities), then get the option prices at every node by backward induction under risk-neutral $q$

3. Solve the portfolio LP
    - By this point, every price on every node is a fixed number, computed and done. The only remaining unknowns are the trading decisions ($x_{in}$, $c^h_{1n}(j,k)$, etc.). Solve

        $$\max_{x,c,p} \,\,\,\, (1-\lambda) \ \mathbb E[W_{N_T}] - \lambda \ \text{CVaR}_{\alpha}(W_{N_T})$$

        using the physical probabilities $p(n)$, subject to the wealth / cash / inventory-balance /position-bound constraints, plus whichever strategy-shape constraints you're imposing if any (protective put / covered call / straddle, ...).

4. Rolling / Receding-horizon re-optimization
    - At $t=0$, run steps 1,2,3, get the decision and execute it.
    - Let one period pass, now at $t=1$ observe the actual realized market return
    - At this new date, rebuild a fresh tree from stage 1 (using updated historical data up to now), re-price derivative (stage 2), re-solve LP (stage 3), then get a new decision and execute it
    - Repeat

5. Evaluation (Out-of-sample check)
    - After solving, find whichever path is closest (Euclidean distance) to what market actually did, and read off that path's node-level decisions to mark-to-market the strategy at realized prices. Roll forward to test on longer period, using period final wealth as the next period initial wealth.
    - This checks whether the built tree was good or not (even if the policy is optimal on that tree, the tree might be a misrepresentation of the market)

**Advantages**
- Reshape historical return distribution
- Empirically strong results
- Regime-adaptive sizing, not a fixed rule
    - Optimizer decides when, how much and which strikes/maturities
- Tunable risk appetite
    - $\lambda$ gives a single and interpretable choice between growth and risk
- Fully interpretable
- Exact optimum from stated objective
    - LP solver finds optimal policy
- Hard constraints are exactly enforced 

**Disadvantages**
- Curse of dimensionality
- Policy is optimal for that one finite tree
    - The tree might be a misrepresentation of the market
- Historical bootstrap dependence
- No transferable policy function
    - Every rebalancing require new solving, cannot learn a policy and evaluate instantly like a trained neural policy
- Option pricing under incompleteness is a modeling choice
    - MEM/entropy measure is one defensible way to pick among the infinitely many risk-neutral measures consistent with no-arbitrage in an incomplete market.

___
___
### **2. Research Overview**


### Deep Hedging

H. Buehler, L. Gonon, J. Teichmann, and B. Wood. Deep hedging. Quantitative Finance, 19(8):1271–1291, 2019.
https://arxiv.org/abs/1802.03042 

*Deep hedging replaces the closed-form replication strategy of models like Black-Scholes with a deep reinforcement learning agent trained directly on simulated price paths of the underlying asset. Given a derivative to hedge, the network learns a trading policy (how much of the underlying to hold at each point in time) that minimizes a chosen risk measure on the resulting hedged profit and loss across many simulated scenarios, rather than matching an analytical hedge ratio.* 

*This makes the approach applicable beyond the frictionless, closed-form setting: it extends naturally to transaction costs, alternative risk objectives, and underlying price dynamics for which no analytical hedging formula exists. The disadvantages are that training requires a large amount of simulated data, and that deep learning's black-box nature means the agent may speculate, take high risk early on, or 'sacrifice' some paths to optimize others.*

**Summary**

*Scenario*

Let {$S_t$}$_{t=0}^{T}$ be a sequence of asset prices (e.g. stock price), let {$\delta_t$}$_{t=0}^{T-1}, \, \delta_t \in [0,1]$ be the number of shares (fractional allowed) of asset $S_t$ hold at time $t$. Suppose you sell a call option with strike $K$ and maturity $T$, and the market risk-free rate $r$ is deterministic. You receive an initial premium $C_0$ and have to deliver a final payoff $P_T = \text{max}(S_T - K)^+$.

*Black-Scholes Framework*

In the simple Black-Scholes framework, with no market friction, the value of a call option at time $t < T$ is the function $C: (S_t, K, \sigma (T - t), r) \rightarrow \mathbb{R}^+$. For replicating (in this case 'delta hedging') the short position on the call option, the number of shares of asset $S_t$ you need to hold at time $t$ is defined as $\delta_t = \frac{\partial C_t}{\partial S_t}= \Phi(d_1) \in [0,1]$, $\Phi(\cdot)$ is the standard normal CDF. 

*Deep Hedging Framework*

Simulate $N$ Monte Carlo paths of the underlying asset $S_t$ with discretization $t=0,1,...,T$. Each path $S_t^i$, for $i=1,...,N$, has its own call option payoff $P_T^i$, and produces its own realized profit & loss: $ \text{PnL}^i = C_0 e^{rT} - P_T^i + \sum_{t=0}^{T-1} \delta_t (S_{t+1}^i - S_t^i) e^{r(T-t)}$. In other words, PnL is equal to the compounded option premium minus the option payoff, plus the compounded profit and loss of each rebalancing steps.


The objective of the deep hedging agent is to learn the optimal trading policy {$\delta_t$}$_{t=0}^{T-1}$ that optimizes a risk measure (e.g. MSE, SMSE, CVaR) on the entire $N$-simulated PnL. The trading policy $\delta_t$ is learned through a neural network, with inputs $\delta_{t-1}$ (the last position), $S_t$ (the current asset price) and $K$ (the strike price), and output activation is sigmoid to ensure $\delta_t \in [0,1]$.


**Sample Code**


**Results**

*Please refer to the file: Project\Deep_Hedging\Simple_Deep_hedging.ipynb*

Observe the following results, assuming the simulated paths of asset price are Black-Scholes. The simulation for training and testing is $N$=1000 Monte-Carlo paths (very small sample size), each with 1000 discretization. The neural network has 2 hidden layer, each with 64 neurons, ReLU activation, Sigmoid output activation and the training is done with 500 epochs.

As we can see, the deep hedging agent converges to Black-Scholes delta hedging.

**Advantages**

- Easily adjustable for transaction costs, market impacts, trading constraints, etc.
- Can add inputs (option surfaces) to the network for multi-instruments hedging.
- Can be adjusted from a hedging prospective to a optimal execution strategy (e.g. rebalancing rule in a covered call strategy).

**Disadvantages**

- Requires lot of training data (simulated).
- Black box: Sometime, the learning agent may 'sacrifice' some paths to optimize others. The hedging agent may speculate, or take high risk early on.

___
### Option Surface Modeling

Hans Buehler, Blanka Horvath, Anastais Kratsios, Yannick Limmer, and Raeid Saqur. Sanos - smooth arbitrage-free non-parametric option surfaces. https://arxiv.org/abs/2601.11209, February 2026.


*SANOS (Smooth strictly Arbitrage-free Non-parametric Option Surfaces) represents call prices as a convex combination of Black-Scholes call kernels anchored at a grid of strikes and variances (observed and extended). The weights, a martingale density, are fit via linear/convex programming to match observed market bid/ask prices; because the combination is convex by construction, the resulting surface is automatically smooth and strictly arbitrage-free, and extends naturally to strikes and expiries beyond those directly quoted.*


**Code & Results**

*Please refer to the file: Project\Vol_Models\SANOS_theory_implementation.ipynb*

___
### Option Surface Simulation

Hans Buehler, Blanka Horvath, Anastais Kratsios. Dysanos - Generative Dynamic Smooth Arbitrage-free Non-parametric Option Surfacess. https://arxiv.org/abs/2608.12587, August 2026.

*DYSANOS learns a mapping from an unconstrained latent space to the martingale densities of SANOS that generate a smooth, arbitrage-free option surface, allowing the surface to be represented by a reduced set of unrestricted parameters. Because this mapping is differentiable end-to-end, it can be trained as a single neural network directly on historical option surfaces. Training then yields a low-dimensional time series describing how the surface evolves. This reduced time series can be modeled jointly with the underlying spot price, allowing realistic joint simulation of the spot and the full option surface going forward.*

*Its main strength is that it produces a smooth, arbitrage-free option surface by construction, rather than as a constraint imposed after fitting. Its main weakness is that the resulting reduced surface state is not directly interpretable, and that static absence of arbitrage does not guarantee absence of arbitrage across trading times, which must be checked separately.*


##### *Surface re-modeling*

The above option surface modeling SANOS can be viewed as the mapping
$$
(q, W; \hat{K}, \hat{T}, \mu) \xrightarrow{SANOS} C_{\tiny SANOS}
$$
where $q$ is the martingale densities (variable to optimized), $W$ is the total volatility of the BS-kernel (subject to non-arbitrage constraints: variance of the terminal distribution can only grow with horizon, not shrink), $\hat{K}$ and $\hat{T}$ are the surface grid, $\mu$ is the smoothing parameter). $C_{\tiny SANOS}$ is a complete option surface.

Since $q$ is subject to linear constraints, it is hard to use ML on it. The idea of ML-SANOS is to create the mapping
$$
\small 
(x) \xrightarrow{\tiny ML-SANOS} 
(\Sigma, W) \xrightarrow{\tiny DLV} (q, W) \xrightarrow{\tiny SANOS} C_{\tiny SANOS}
$$
where $x$ and the decoder function ML-SANOS are trained from real market option surfaces. The objective function are the sum of losses from SANOS option surface against bid-ask and mid prices.

##### *Intermediary Results: Option Surface Time Series*


In this new parametrization, $x$ lives in the unrestricted space and is therefore well suited for ML/AI based learning. Note that $(x)$ is already a valid time series which can be decoded to obtain a valid option surface at every time steps.

For efficient time series analysis, reduce the dimension of $(x) \in \mathbb{R}^{\tiny FullDim}$ to $h \in \mathbb{R}^{\tiny LowDim}$

$$
(h)^{\tiny LowDim} \xrightarrow{\tiny NQ} (x)^{\tiny FullDim} 
$$

After training, we are left with 2 things:
- A time serie {${h_t}$}$_{t=1}^{N_{\tiny training}}$, $h_t \in \mathbb{R}^{\tiny LowDim}$ for each training day $t$
- A single shared decoder $\theta$ from $NQ_{\theta}$ (that leads to a valid SANOS option surface), valid across all days

Note that in practice, we train the whole thing at once ($x$ and $h$ are train together)

##### *Surface Simulation*

Assume we are given historic samples of ML-SANOS surface states $\tilde{h}_t = (\tilde{h}^1_t, \ldots, \tilde{h}^{n_h}_t) \in \mathbb{R}^{n_h}$ for each historic date $t \in {t_1, \ldots, t_{n_t}}$. We also observe log-spot $\tilde{s}_t := \log S_t$. We use the tilde to distinguish real observed data from simulated data.

<br>

Model a time serie $h$ as followed (**baseline model, potential for further research**):

*Continuous-time PCA-AR(1)*
$$
dh_t = \kappa (m - h_t) \ dt + \Sigma_h \ dW_t^h
$$
for $m \in \mathbb{R}^{n_h}$, $\kappa \in \mathbb{R}^{n_h, n_h}$, $\Sigma_h \in \mathbb{R}^{n_h,n_\alpha}$ and a Brownian motion $W_t^h \in \mathbb{R}^{n_\alpha}$. $n_h$ is the dimention of thre vector $h_t$, $n_{\alpha}$ is the number of PCA factors (or no PCA and analyse full dimension of $h$).

<br>

Now model the log-spot price time serie $s_t$

*Log-spot* (fitted afterward)
$$
ds_t = \mu dt + \beta^{\prime} dW_t^h + \varsigma dW_t^s
$$
where $\mu \in \mathbb{R}$ is an intercept, $\beta \in \mathbb{R}^{n_h}$, $\varsigma \in \mathbb{R}$ and $dW_t^s \in \mathbb{R}$ is an independant Brownian motion. This preserves the autonomous surface dynamics and captures contemporaneous spot–surface dependence through $\beta$.

*Results*

- First, we have the historical time serie {$\tilde{h}_t$}, which can be decoded to complete valid option surfaces at each $t$.

- Second, from this time serie, we can simulate $h_t$ (which can be decoded to option surface). Since the option surfaces are normalized (and more), we can simulate spot prices based on the simulated option surfaces, through a defined (or fitted) dependance.

*Simulation* (**base case example**)

1. Draw randomly $h_0$ from ${\tilde h}_{t\in{1,\dots,n_t}}$ or the invariant distribution.

2. Generate the surface path using the Euler discretization
    $$
    h_{t+dt} = h_t + \kappa(m - h_t)\Delta_t + \Sigma_h Z_t\sqrt{\Delta_t}
    $$
    where $\Delta_t$ is the normalized time increment. The noise term $Z_t$ blends a resampled (standardized) historical PCA innovation with an independent Gaussian innovation, weighted by $b\in[0,1]$: $b=0$ is pure empirical resampling, $b=1$ is pure Gaussian with the same fitted covariance (once scaled by $\Sigma_h$).

3. Conditional on the complete surface path, generate log-spot 
    $$
    ds_t = \mu dt + \beta^{\prime} dW_t^h + \varsigma dW_t^s
    $$
    where $\mu$ is the drift, $\beta$ represent the dependance with the option surface, and $\varsigma$ is the diffusion of the spot-specific shoc. (e.g. $\mu$ and $\varsigma$ constant, $\beta=0$, is an independant Black-Scholes diffusion process)

___
___
### **3. Research Proposal**


**Research Plan**
1.	Literature review. Problem definition. Data acquisitions & cleaning.
2.	Fit the DYSANOS option surface model to historical option surfaces to obtain a reduced-dimension latent time series.
3.	Simulate jointly asset spot price and reduced option surface with a block bootstrap of historical filtered returns. Keep a parametric mean reversion in the simulated reduced option surfaces.
4.	Train a deep hedging reinforcement-learning agent on the joint simulated paths to learn optimal options rebalancing (timing, sizing, strike selection) that maximizes a trade-off between expected wealth and risk (e.g. CVaR). The agent has the whole reduced option surface as an additional input, and can trade options.


**Potential Extensions**
- Simulate jointly the asset spot price and reduced option surfaces through dependence-based time series models/generative models.
- Add realistic constraints to the deep hedging agent such as liquidity, transaction costs, risk/exposure limits, etc.
- De-Americanized options: the above theory is about European options. Trading American options requires an adjustment (called de-Americanization). Note that covered positions (such as covered call) require a less strict adjustment (less risky).
